# SawitSense — M1 FFB ripeness training (Kaggle GPU)
Trains EfficientNet-B0 on **D1** Kamal et al. (2024, CC BY 4.0, Indonesia) and tests on **D2** Rosbi (2024, CC BY 4.0, Malaysia).
Outputs in `/kaggle/working`: `m1_ripeness.onnx`, `m1_ripeness.json`, `m1_metrics.json`.

**Settings → Accelerator: GPU T4 ×2 or P100. Internet: ON.**

In [ ]:
!nvidia-smi -L || echo 'No GPU — enable it in Settings > Accelerator'
!pip -q install onnx onnxscript onnxruntime

In [ ]:
import hashlib, os, subprocess, zipfile, pathlib
data = pathlib.Path('/kaggle/tmp/data'); data.mkdir(parents=True, exist_ok=True)
D1_URL = 'https://data.mendeley.com/public-files/datasets/424y96m6sw/files/cae49a93-5785-4873-a347-bff2a276f62c/file_downloaded'
D1_SHA = 'c3cf88db3194fedd24894a35e27fada3841977e1a09a405a4e064db02ebde0c5'
D2_URL = 'https://zenodo.org/api/records/11114885/files/MunirahRosbi/Outdoor-Tenera-Oil-Palm-Fruit-Image-v1.zip/content'
def fetch(url, dst):
    if not dst.exists():
        subprocess.run(['curl', '-sSL', '--retry', '5', '-o', str(dst), url], check=True)
    return dst
d1 = fetch(D1_URL, data / 'd1.zip')
assert hashlib.sha256(d1.read_bytes()).hexdigest() == D1_SHA, 'D1 checksum mismatch'
d2 = fetch(D2_URL, data / 'd2.zip')
for z, out in ((d1, data / 'd1'), (d2, data / 'd2')):
    if not out.exists():
        zipfile.ZipFile(z).extractall(out)
# D1 may contain nested zips — extract them too
for inner in (data / 'd1').rglob('*.zip'):
    zipfile.ZipFile(inner).extractall(inner.with_suffix(''))
!du -sh /kaggle/tmp/data/*; find /kaggle/tmp/data/d1 -maxdepth 3 -type d | head -30

In [ ]:
%%writefile /kaggle/working/preprocess.py
"""Shared M1 image preprocessing — the SAME code runs in training/evaluation (train_m1.py) and in the
API (inference.py), so reported metrics are the serving-path metrics (S-16).
Pure PIL + numpy, no relative imports (train_m1.py imports it as a plain module)."""
from __future__ import annotations

import numpy as np
from PIL import Image, ImageOps

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
BASE_SHORT = 288   # stage 1: normalise any upload (phone/drone, up to 5 MB) to a 288-px short side
RESIZE = 256       # stage 2: short side 256 …
SIZE = 224         # … centre crop 224


def base_image(img: Image.Image) -> Image.Image:
    """EXIF-orient, RGB, downscale so the short side is BASE_SHORT (bilinear)."""
    try:
        img = ImageOps.exif_transpose(img)
    except Exception:  # noqa: BLE001 — some phone JPEGs carry malformed EXIF
        pass
    img = img.convert("RGB")
    img.thumbnail((BASE_SHORT * 4, BASE_SHORT * 4))
    w, h = img.size
    s = BASE_SHORT / min(w, h)
    return img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)


def eval_crop(img288: Image.Image, resize: int = RESIZE, size: int = SIZE) -> Image.Image:
    w, h = img288.size
    s = resize / min(w, h)
    img = img288.resize((max(size, round(w * s)), max(size, round(h * s))), Image.BILINEAR)
    w, h = img.size
    left, top = (w - size) // 2, (h - size) // 2
    return img.crop((left, top, left + size, top + size))


def to_array(crop: Image.Image, mean=MEAN, std=STD) -> np.ndarray:
    a = (np.asarray(crop, dtype=np.float32) / 255.0 - np.array(mean, np.float32)) / np.array(std, np.float32)
    return a.transpose(2, 0, 1).astype(np.float32)        # CHW


def eval_array(img: Image.Image) -> tuple[np.ndarray, Image.Image]:
    """Full serving path for an opened image: returns (CHW float32, 224-px RGB crop)."""
    crop = eval_crop(base_image(img))
    return to_array(crop), crop


In [ ]:
%%writefile /kaggle/working/train_m1.py
"""M1 — FFB ripeness classifier (EfficientNet-B0 transfer learning) → ONNX for the SawitSense API.

Data (D16): D1 Kamal et al. (2024), Indonesia, CC BY 4.0 → train/val/test.
            D2 Rosbi (2024), Malaysia, CC BY 4.0 → cross-country test (+ optional head fine-tune).
Classes (harmonised, PLAN M1): unripe · underripe · ripe · overripe  (D2 Empty/Damaged excluded).

Two stages, same script on a laptop CPU or a Kaggle GPU:
  1. Frozen backbone: extract pooled features (with flip/crop augmentation), train a linear head.
  2. Optional fine-tune of the last blocks (--finetune-epochs N; use on GPU).
Outputs (to --out, default 03_software/models/):
  m1_ripeness.onnx   — inputs "input" [N,3,224,224]; outputs "logits", "features" [N,1280,7,7]
  m1_ripeness.json   — classes, preprocessing, classifier weights (for CAM), metrics
  m1_metrics.json    — full metrics incl. confusion matrices
Examples:
  python train_m1.py --d1 ../../data/raw/d1 --d2 ../../data/raw/d2                     # CPU, stage 1 only
  python train_m1.py --d1 /kaggle/input/... --d2 /kaggle/input/... --device cuda --finetune-epochs 8 --out /kaggle/working
"""
from __future__ import annotations

import argparse
import hashlib
import sys
import json
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# Shared serving preprocessing (S-16): repo path first, else a copy next to this script (Kaggle).
sys.path[:0] = [str(Path(__file__).resolve().parents[2] / "src" / "app"), str(Path(__file__).resolve().parent)]
import preprocess as PP  # noqa: E402

CLASSES = ["unripe", "underripe", "ripe", "overripe"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
VERSION = "m1-effb0-0.1"


def map_class(path: Path) -> str | None:
    """Map a folder name from D1/D2/D3 to a harmonised class (order matters)."""
    s = "/".join(p.lower() for p in path.parts[-3:-1])
    if re.search(r"damaged|empty", s):
        return None
    if re.search(r"overripe|over[-_ ]ripe|decay", s):
        return "overripe"
    if re.search(r"partial|underripe|under[-_ ]ripe|mengkal", s):
        return "underripe"
    if re.search(r"immature|unripe|mentah", s):
        return "unripe"
    if re.search(r"ripe|masak|mature", s):
        return "ripe"
    return None


def dhash(path: Path, size: int = 8) -> int:
    """Difference hash (64-bit) for near-duplicate detection (S-17)."""
    img = Image.open(path)
    img.draft("L", (size * 8, size * 8))          # fast JPEG decode at reduced size
    g = np.asarray(img.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


def dedupe_test(test, train, max_dist: int = 6):
    """Drop test images whose dHash is within max_dist bits of any train image (same photograph re-used
    across the provided D2 split — S-17). Returns (kept, dropped_pairs)."""
    th = [(it, dhash(it[0])) for it in train]
    kept, dropped = [], []
    for it in test:
        h = dhash(it[0])
        twin = next(((t, bin(h ^ x).count("1")) for t, x in th if bin(h ^ x).count("1") <= max_dist), None)
        if twin:
            dropped.append({"test": str(Path(*it[0].parts[-2:])), "train_twin": str(Path(*twin[0][0].parts[-2:])),
                            "hamming": twin[1]})
        else:
            kept.append(it)
    return kept, dropped


def split_of(path: Path) -> str | None:
    s = "/".join(p.lower() for p in path.parts)
    for name, keys in (("test", ("test",)), ("val", ("val", "valid")), ("train", ("train",))):
        if any(re.search(rf"(^|/|_|-){k}", s) for k in keys):
            return name
    return None


def scan(root: Path) -> list[tuple[Path, int, str | None]]:
    items = []
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() in IMG_EXT and not p.name.startswith("."):
            c = map_class(p)
            if c:
                items.append((p, CLASSES.index(c), split_of(p.relative_to(root))))
    return items


def stratified(items, seed=42, val=0.15, test=0.15):
    rng = random.Random(seed)
    by = {}
    for it in items:
        by.setdefault(it[1], []).append(it)
    out = {"train": [], "val": [], "test": []}
    for lst in by.values():
        rng.shuffle(lst)
        n = len(lst); nt, nv = int(n * test), int(n * val)
        out["test"] += lst[:nt]; out["val"] += lst[nt:nt + nv]; out["train"] += lst[nt + nv:]
    return out


class ImgDS(Dataset):
    def __init__(self, items, tf, cache: Path | None = None):
        self.items, self.tf, self.cache = items, tf, cache

    def __len__(self):
        return len(self.items)

    def _open(self, p: Path) -> Image.Image:
        # Cached base image (288-px short side), stored losslessly so train/eval/serving see the same pixels.
        if self.cache:
            cp = self.cache / (hashlib.md5(str(p).encode()).hexdigest() + ".png")
            if cp.exists():
                return Image.open(cp).convert("RGB")
        img = PP.base_image(Image.open(p))
        if self.cache:
            img.save(cp)
        return img

    def __getitem__(self, i):
        p, y, _ = self.items[i]
        return self.tf(self._open(p)), y


def tfs(train: bool):
    if train:
        return transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
                                   transforms.ColorJitter(0.15, 0.15, 0.1, 0.02), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    # Evaluation = the exact serving path (shared preprocess module)
    return eval_tf


def eval_tf(img):   # module-level (picklable for DataLoader workers)
    return torch.from_numpy(PP.to_array(PP.eval_crop(img)))


@torch.no_grad()
def extract(backbone, items, tf, device, bs, workers, cache, passes=1):
    feats, ys = [], []
    for _ in range(passes):
        dl = DataLoader(ImgDS(items, tf, cache), batch_size=bs, num_workers=workers)
        for x, y in dl:
            f = backbone.avgpool(backbone.features(x.to(device))).flatten(1)
            feats.append(f.cpu()); ys.append(y)
    return torch.cat(feats), torch.cat(ys)


def train_head(head, X, y, Xv, yv, epochs=300, lr=1e-3, wd=1e-4, class_weight=None):
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.CrossEntropyLoss(weight=class_weight)
    best, best_state = -1, None
    for ep in range(epochs):
        head.train()
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 256):
            idx = perm[i:i + 256]
            opt.zero_grad(); lossf(head(X[idx]), y[idx]).backward(); opt.step()
        if ep % 10 == 9 or ep == epochs - 1:
            head.eval()
            with torch.no_grad():
                acc = (head(Xv).argmax(1) == yv).float().mean().item()
            if acc > best:
                best, best_state = acc, {k: v.clone() for k, v in head.state_dict().items()}
    head.load_state_dict(best_state)
    return best


def metrics(y_true, y_pred, labels=range(len(CLASSES))):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    cm = np.zeros((len(CLASSES), len(CLASSES)), int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    present = [c for c in labels if cm[c].sum() > 0]
    f1s, per = [], {}
    for c in present:
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; fn = cm[c].sum() - tp
        prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        f1s.append(f1); per[CLASSES[c]] = {"precision": round(prec, 3), "recall": round(rec, 3), "f1": round(f1, 3), "n": int(cm[c].sum())}
    return {"n": int(len(y_true)), "accuracy": round(float((y_true == y_pred).mean()), 4),
            "macro_f1_present_classes": round(float(np.mean(f1s)), 4),
            "within_one_class_accuracy": round(float((np.abs(y_true - y_pred) <= 1).mean()), 4),
            "per_class": per, "confusion_matrix": {"labels": CLASSES, "rows_true_cols_pred": cm.tolist()}}


class Exportable(nn.Module):
    def __init__(self, net):
        super().__init__(); self.net = net

    def forward(self, x):
        f = self.net.features(x)
        return self.net.classifier(self.net.avgpool(f).flatten(1)), f


def predict(net, items, device, bs, workers, cache):
    net.eval(); preds = []
    with torch.no_grad():
        for x, _ in DataLoader(ImgDS(items, tfs(False), cache), batch_size=bs, num_workers=workers):
            preds.append(net(x.to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy() if preds else np.array([], int)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--d1", type=Path, help="extracted D1 root (omit = PROVISIONAL model trained on D2 train only)")
    ap.add_argument("--d2", type=Path, help="D2 root (FFBtest/ FFBtrain/)")
    ap.add_argument("--out", type=Path, default=Path(__file__).resolve().parents[2] / "models")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--aug-passes", type=int, default=2, help="augmented feature passes for the head (stage 1)")
    ap.add_argument("--finetune-epochs", type=int, default=0)
    ap.add_argument("--max-per-class", type=int, default=0, help="cap training images per class (quick runs)")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()
    torch.manual_seed(a.seed); random.seed(a.seed); np.random.seed(a.seed)
    a.out.mkdir(parents=True, exist_ok=True)
    cache = a.out / "_imgcache"; cache.mkdir(exist_ok=True)
    t0 = time.time()

    d2 = scan(a.d2) if a.d2 and a.d2.exists() else []
    d2_test = [it for it in d2 if it[2] == "test"]
    d2_train = [it for it in d2 if it[2] == "train"]
    d2_dupes = []
    if d2_test and d2_train:
        n0 = len(d2_test)
        d2_test, d2_dupes = dedupe_test(d2_test, d2_train)
        d2 = d2_train + d2_test
        print(f"D2 dedupe: dropped {len(d2_dupes)} of {n0} test images with a train twin (dHash <= 6)", flush=True)
        for d in d2_dupes:
            print("   ", d, flush=True)
    print("D2:", dict(Counter(CLASSES[i[1]] for i in d2)), "test", len(d2_test), "train", len(d2_train), flush=True)
    provisional = False
    d1 = scan(a.d1) if a.d1 else []
    if not d1:
        if not (d2_train and d2_test):
            raise SystemExit("No D1 images and no D2 train/test split found")
        # PROVISIONAL: D1 unavailable on this machine -> train on Malaysian D2 train, test on D2 test.
        provisional = True
        tr = stratified(d2_train, a.seed, val=0.15, test=0.0)
        sp = {"train": tr["train"], "val": tr["val"], "test": d2_test}
        split_src = "PROVISIONAL: D2 train (85/15 train/val) -> D2 test; D1 not used"
        d2, d2_test, d2_train = [], [], []
    elif all(s for _, _, s in d1):
        sp = {k: [it for it in d1 if it[2] == k] for k in ("train", "val", "test")}
        split_src = "dataset-provided split"
    else:
        sp = stratified(d1, a.seed); split_src = "stratified 70/15/15 (seed 42)"
    if not provisional and not sp["val"]:
        tr = stratified(sp["train"], a.seed, val=0.15, test=0.0); sp["train"], sp["val"] = tr["train"], tr["val"]
    if a.max_per_class:
        cnt, keep = Counter(), []
        for it in sp["train"]:
            if cnt[it[1]] < a.max_per_class:
                keep.append(it); cnt[it[1]] += 1
        sp["train"] = keep
    print({k: dict(Counter(CLASSES[i[1]] for i in v)) for k, v in sp.items()}, split_src, flush=True)

    net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1).to(a.device).eval()
    head = nn.Linear(1280, len(CLASSES))

    # Stage 1 — frozen features + linear head
    Xtr, ytr = extract(net, sp["train"], tfs(True), a.device, a.bs, a.workers, cache, passes=a.aug_passes)
    Xtr_c, ytr_c = extract(net, sp["train"], tfs(False), a.device, a.bs, a.workers, cache)
    Xtr, ytr = torch.cat([Xtr, Xtr_c]), torch.cat([ytr, ytr_c])
    Xv, yv = extract(net, sp["val"], tfs(False), a.device, a.bs, a.workers, cache)
    counts = torch.bincount(ytr, minlength=len(CLASSES)).float()
    present = counts > 0
    cw = torch.where(present, counts.sum() / (present.sum() * counts.clamp(min=1)), torch.zeros_like(counts))
    # classes absent from training (e.g. underripe in D2-only mode) get weight 0, otherwise label smoothing
    # would push probability mass toward them
    val_acc = train_head(head, Xtr, ytr, Xv, yv, class_weight=cw)
    print(f"stage1 val acc {val_acc:.4f}  ({time.time() - t0:.0f}s)", flush=True)
    net.classifier[1] = head.to(a.device)

    # Stage 2 — optional fine-tune (GPU recommended)
    if a.finetune_epochs:
        for p in net.features[:-3].parameters():
            p.requires_grad = False
        opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=3e-4, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, a.finetune_epochs)
        lossf = nn.CrossEntropyLoss(weight=cw.to(a.device), label_smoothing=0.05 if bool(present.all()) else 0.0)
        dl = DataLoader(ImgDS(sp["train"], tfs(True), cache), batch_size=a.bs, shuffle=True, num_workers=a.workers)
        best, best_state = val_acc, {k: v.clone() for k, v in net.state_dict().items()}
        def train_mode():
            net.train()
            for m in net.features[:-3].modules():   # frozen blocks: keep BatchNorm statistics fixed
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()
        for ep in range(a.finetune_epochs):
            train_mode()
            for x, y in dl:
                opt.zero_grad(); lossf(net(x.to(a.device)), y.to(a.device)).backward(); opt.step()
            sched.step()
            acc = float((predict(net, sp["val"], a.device, a.bs, a.workers, cache) == np.array([i[1] for i in sp["val"]])).mean())
            print(f"finetune ep{ep + 1} val acc {acc:.4f}", flush=True)
            if acc > best:
                best, best_state = acc, {k: v.clone() for k, v in net.state_dict().items()}
        net.load_state_dict(best_state); val_acc = best
    net.eval()

    version = VERSION + ("-provisional" if provisional else "")
    res = {"model_version": version, "provisional": provisional, "split": split_src, "val_accuracy": round(val_acc, 4),
           ("d2_malaysia_test" if provisional else "d1_test"): metrics([i[1] for i in sp["test"]], predict(net, sp["test"], a.device, a.bs, a.workers, cache))}
    if d2:
        res["d2_malaysia_all_zero_shot"] = metrics([i[1] for i in d2], predict(net, d2, a.device, a.bs, a.workers, cache))
    if d2_test:
        res["d2_malaysia_test_zero_shot"] = metrics([i[1] for i in d2_test], predict(net, d2_test, a.device, a.bs, a.workers, cache))
    if d2_train and d2_test:
        # Variant: adapt only the head with Malaysian train images (keeps the D1 backbone), test on D2 test.
        Xd, yd = extract(net, d2_train, tfs(True), a.device, a.bs, a.workers, cache, passes=a.aug_passes)
        head2 = nn.Linear(1280, len(CLASSES)); head2.load_state_dict(net.classifier[1].state_dict())
        Xmix = torch.cat([Xtr, Xd, Xd]); ymix = torch.cat([ytr, yd, yd])   # D2 up-weighted ×2
        train_head(head2.cpu(), Xmix, ymix, Xv.cpu(), yv, epochs=150)
        Xt2, yt2 = extract(net, d2_test, tfs(False), a.device, a.bs, a.workers, cache)
        res["d2_malaysia_test_after_head_finetune"] = metrics(yt2.numpy(), head2(Xt2).argmax(1).numpy())
    res["d2_test_duplicates_removed"] = d2_dupes
    res["train_seconds"] = round(time.time() - t0)
    res["device"] = a.device
    res["finetune_epochs"] = a.finetune_epochs
    res["train_counts"] = dict(Counter(CLASSES[i[1]] for i in sp["train"]))

    # Export ONNX (logits + features for CAM)
    net_cpu = Exportable(net.cpu()).eval()
    onnx_path = a.out / "m1_ripeness.onnx"
    torch.onnx.export(net_cpu, torch.randn(1, 3, 224, 224), str(onnx_path), input_names=["input"],
                      output_names=["logits", "features"], dynamic_axes={"input": {0: "n"}, "logits": {0: "n"}, "features": {0: "n"}},
                      opset_version=17, dynamo=False)
    fc = net.classifier[1]
    meta = {"model_version": version, "provisional": provisional, "classes": CLASSES, "input_size": 224, "resize": 256, "mean": MEAN, "std": STD,
            "fc_weight": fc.weight.detach().cpu().numpy().round(6).tolist(), "fc_bias": fc.bias.detach().cpu().numpy().round(6).tolist(),
            "architecture": "EfficientNet-B0 (ImageNet) transfer learning", "explainability": "CAM from final linear layer (= Grad-CAM for GAP-linear head)",
            "data": ("PROVISIONAL: train/test D2 Rosbi 2024 Malaysia (CC BY 4.0); no underripe class in D2" if provisional
                     else "Train D1 Kamal et al. 2024 (CC BY 4.0); test D2 Rosbi 2024 Malaysia (CC BY 4.0)"),
            "metrics_summary": {k: v["accuracy"] for k, v in res.items() if isinstance(v, dict) and "accuracy" in v}}
    (a.out / "m1_ripeness.json").write_text(json.dumps(meta))
    (a.out / "m1_metrics.json").write_text(json.dumps(res, indent=2))
    print(json.dumps(meta["metrics_summary"], indent=2), f"\nsaved {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)")

    # Serving-path evaluation (S-16): exported ONNX + the API's own preprocessing on the original files.
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    serve_sets = {"d2_malaysia_test": sp["test"]} if provisional else {"d1_test": sp["test"]}
    if d2_test:
        serve_sets["d2_malaysia_test_zero_shot"] = d2_test
    res["serving_path"] = {}
    for name, items in serve_sets.items():
        preds = []
        for pth, _, _ in items:
            x, _crop = PP.eval_array(Image.open(pth))
            preds.append(int(sess.run(["logits"], {"input": x[None]})[0].argmax()))
        res["serving_path"][name] = metrics([i[1] for i in items], preds)
        print(f"serving path {name}: acc {res['serving_path'][name]['accuracy']}", flush=True)
    meta["metrics_summary_serving_path"] = {k: v["accuracy"] for k, v in res["serving_path"].items()}
    (a.out / "m1_ripeness.json").write_text(json.dumps(meta))
    (a.out / "m1_metrics.json").write_text(json.dumps(res, indent=2))

    # Sanity check: ONNX matches PyTorch
    x = torch.randn(2, 3, 224, 224)
    lo, fo = ort.InferenceSession(str(onnx_path)).run(None, {"input": x.numpy()})
    lt, ft = net_cpu(x)
    print("onnx max |diff| logits", float(np.abs(lo - lt.detach().numpy()).max()))


if __name__ == "__main__":
    main()


In [ ]:
!cd /kaggle/working && python train_m1.py --d1 /kaggle/tmp/data/d1 --d2 /kaggle/tmp/data/d2 --device cuda --finetune-epochs 8 --workers 4 --out /kaggle/working

In [ ]:
import json; m = json.load(open('/kaggle/working/m1_metrics.json'))
for k, v in m.items():
    if isinstance(v, dict) and 'accuracy' in v:
        print(f"{k:40s} acc={v['accuracy']:.3f} macroF1={v['macro_f1_present_classes']:.3f} within1={v['within_one_class_accuracy']:.3f} n={v['n']}")
!rm -rf /kaggle/working/_imgcache; ls -la /kaggle/working